# 10 · Model A: H3 cell classifier (GPU)

**Idea (after Flipkart, COLING 2025):** treat geocoding as classification. Cut the city into H3 hexagons and train a text encoder to say which hexagon an address belongs to. One classification head per resolution (7, 8, 9: about 5 sq km, 0.7 sq km, 0.1 sq km), trained together.

```
address text -> encoder -> vector -> head r7  (coarse)
                                   -> head r8
                                   -> head r9  (fine)  -> centre of the chosen cell
```

**Prediction** walks down: best r7 cell, then the best r8 cell inside it, then the best r9 cell inside that.

**What to expect:** strong on warm test data. On cold test data it *cannot* be right, because it can only answer with cells seen in training. That gap is one of the benchmark's headline results, not a bug.

**Encoder:** `google/muril-base-cased` (Google's BERT for Indian languages, including Hindi and Telugu written in Roman letters; Apache-2.0).

**Runtime:** about 35 minutes on a T4 for 3 epochs over 300,000 addresses. **Needs:** notebook 06. The code path was checked on CPU with a tiny random network (`tests/check_models.py`); it has not been trained on a GPU yet, so watch the first loss lines.

In [ ]:
# Setup (the same cell opens every notebook): fetch the project code, install what is missing, find the data folder.
import os, sys, subprocess
REPO = 'SiddharthaRaja/Navigate'
PIP  = ['h3', 'rapidfuzz', 'transformers>=4.44']
on_cloud = os.path.exists('/content') or os.path.exists('/kaggle/working')
if on_cloud:
    CODE = '/kaggle/working/Navigate-code' if os.path.exists('/kaggle/working') else '/content/Navigate'
    if not os.path.exists(f'{CODE}/bench'):
        token = os.environ.get('GH_TOKEN', '')
        try:                                   # only needed while the GitHub repo is private
            from google.colab import userdata; token = token or userdata.get('GH_TOKEN')
        except Exception: pass
        try:
            from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret('GH_TOKEN')
        except Exception: pass
        url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
        r = subprocess.run(['git', 'clone', '--depth', '1', url, CODE], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        assert r.returncode == 0, 'Could not fetch the code. If the repo is private, add a GitHub token as the secret GH_TOKEN.'
    else:
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'])
    if PIP: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PIP], check=True)
else:
    CODE = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
sys.path.insert(0, CODE)
import numpy as np, pandas as pd
from bench import env
P = env.setup('hyderabad')
print('running on', P['where'], '| data folder:', P['root'])

import torch
assert torch.cuda.is_available() or P['where'] == 'local', 'No GPU: Runtime > Change runtime type > T4 GPU (Colab) or Settings > Accelerator (Kaggle)'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU trial run)')

In [ ]:
from bench import io, metrics as mt
from models import common, h3_classifier as hc
common.set_seed(7)
ENCODER, EPOCHS, RES = 'google/muril-base-cased', 3, (7, 8, 9)
TRAIN_ROWS = None                     # e.g. 20_000 for a 3-minute trial run
train, val, test = common.load_bench(P['bench'], TRAIN_ROWS)
cells = hc.build_cells(train, RES)
print(f"{len(train):,} training addresses | cells per resolution:", {r: len(v) for r, v in cells['vocab'].items()})

## Train
The loss is the sum of three cross-entropies. It should fall steadily; validation error is printed after each epoch.

In [ ]:
model, tok = hc.train_model(train, cells, encoder_name=ENCODER, epochs=EPOCHS, batch_size=64, val=val.head(2000))

## Predict the test sets and save

In [ ]:
pred = hc.predict(model, tok, test, cells)
io.save_run(P['runs'], 'h3_classifier', pred, about=f'Text encoder ({ENCODER}) with H3 classification heads at resolutions {RES}; hierarchical decoding.')
_, res = mt.evaluate(test, pred, 'h3_classifier', slices=False)
print(mt.leaderboard(res).to_string())
print(f"speed: {pred.latency_ms.median():.2f} ms per address (batched) | weights: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

## Confidence and abstaining
Gibberish rows should get low confidence. This shows how confidence separates them, which sets the `min_conf` threshold for saying "no answer".

In [ ]:
c = pred.merge(test[['id', 'abstain', 'split']], on='id')
print(c.groupby('abstain').conf.describe()[['count', '25%', '50%', '75%']].round(3).to_string())
torch.save({'state': model.state_dict(), 'cells': cells, 'encoder': ENCODER}, f"{P['runs']}/h3_classifier/model.pt")

## Share the outputs
Packs only the files this notebook created into one zip in `My Drive > Navigate > data > share`. Download that one file to share or back up the results.

In [ ]:
env.pack_outputs(P, '10' + '_outputs', [f"{P['runs']}/h3_classifier"])